# Assignment 6 — Colab A100 hybrid LoRA train

Trains Instruct + LoRA on the **hybrid** prepared set (public + skill synth) using Unsloth on an A100.

## Before you run
1. Runtime → Change runtime type → **A100** (Colab Pro).
2. On your laptop, pack and upload prepared data to Google Drive:

```bash
cd /path/to/USU-LLM-Class-F26
mkdir -p ~/usu-a6-upload && tar -czf ~/usu-a6-upload/a6_prepared_hybrid.tgz \
  data/assignment6/prepared_hybrid \
  data/assignment6/qa/hybrid_train.jsonl \
  data/assignment6/qa/hybrid_dev.jsonl
# Upload a6_prepared_hybrid.tgz to Drive as MyDrive/usu-a6/a6_prepared_hybrid.tgz
```

3. Have a Hugging Face token with access to `meta-llama/Llama-3.2-1B-Instruct`.
4. Push this branch to GitHub first so Colab can clone it.


In [ ]:
# 1) GPU check
!nvidia-smi
import torch

assert torch.cuda.is_available(), "Enable a GPU runtime (A100 preferred)"
print(
    torch.cuda.get_device_name(0),
    "VRAM_GiB=",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
)

In [ ]:
# 2) Mount Drive + clone course repo (code only; data from tarball)
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_TGZ = "/content/drive/MyDrive/usu-a6/a6_prepared_hybrid.tgz"  # change if needed
REPO_URL = "https://github.com/IMJONEZZ/USU-LLM-Class-F26.git"
BRANCH = "talha_shahzad_6"

!rm -rf /content/USU-LLM-Class-F26
!git clone --branch {BRANCH} --depth 1 {REPO_URL} /content/USU-LLM-Class-F26
%cd /content/USU-LLM-Class-F26

assert Path(DRIVE_TGZ).is_file(), f"Upload missing: {DRIVE_TGZ}"
!mkdir -p data/assignment6
!tar -xzf "{DRIVE_TGZ}" -C /content/USU-LLM-Class-F26
!ls -la data/assignment6/prepared_hybrid data/assignment6/qa/hybrid_train.jsonl

In [ ]:
# 3) Install training stack (Unsloth)
!pip -q install -U pip
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install peft trl accelerate bitsandbytes datasets "transformers==4.57.6"
!pip -q install evaluate rouge-score bert-score

In [ ]:
# 4) Hugging Face login (gated Llama Instruct)
import getpass
import os

from huggingface_hub import login

token = os.environ.get("HF_TOKEN") or getpass.getpass("HF token: ")
login(token=token)
os.environ["HF_TOKEN"] = token

In [ ]:
# 5) Feasibility then train (same CLI as local; fresh Colab output dirs)
%cd /content/USU-LLM-Class-F26
!rm -rf data/assignment6/feasibility_hybrid_colab data/assignment6/train_hybrid_colab

!python -m src.trainer feasibility \
  --prepared data/assignment6/prepared_hybrid \
  --output data/assignment6/feasibility_hybrid_colab

!python -m src.trainer train \
  --prepared data/assignment6/prepared_hybrid \
  --feasibility data/assignment6/feasibility_hybrid_colab \
  --output data/assignment6/train_hybrid_colab \
  --epochs 3

!ls -la data/assignment6/train_hybrid_colab/best
!test -f data/assignment6/train_hybrid_colab/success.json && echo TRAIN_OK

In [ ]:
# 6) Copy adapter + training.json back to Drive
import json
import shutil
from pathlib import Path

src = Path("data/assignment6/train_hybrid_colab/best")
dst = Path("/content/drive/MyDrive/usu-a6/adapters/hybrid_colab_a100")
dst.parent.mkdir(parents=True, exist_ok=True)
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)
shutil.copy("data/assignment6/train_hybrid_colab/training.json", dst / "training.json")
shutil.copy("data/assignment6/train_hybrid_colab/success.json", dst / "success.json")
meta = json.loads((dst / "run.json").read_text()) if (dst / "run.json").exists() else {}
print("Saved to", dst)
print(
    "checkpoint_epoch",
    meta.get("checkpoint_epoch"),
    "dev_loss",
    meta.get("checkpoint_dev_loss"),
)
print(
    "On your laptop: copy that folder to data/assignment6/adapters/hybrid_colab_a100 "
    "then re-run graded-10 eval against public_only / hybrid_v1."
)